# Lesson 12B: Variational Autoencoder - Practical

## Generating Handwritten Digits

**Case Study**: Train a VAE to learn a smooth latent space of digit images, enabling generation of new digits and interpolation between numbers.

**Key Difference from AE**: VAE learns a probabilistic latent space with KL divergence regularization.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits

torch.manual_seed(42)
np.random.seed(42)
print('✅ Loaded!')

## Step 1: Define VAE Architecture

**VAE Loss** = Reconstruction Loss + KL Divergence

KL divergence pushes latent distribution toward N(0, I).

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=32, latent_dim=8):
        super().__init__()
        # Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder
        self.fc2 = nn.Linear(latent_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, input_dim)
    
    def encode(self, x):
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick: z = μ + σ * ε"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps
    
    def decode(self, z):
        h = F.relu(self.fc2(z))
        return torch.sigmoid(self.fc3(h))
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

vae = VAE(input_dim=64, hidden_dim=32, latent_dim=8)
print('✅ VAE architecture defined')
print(f'   Parameters: {sum(p.numel() for p in vae.parameters())}')

## Step 2: Load and Prepare Data

In [ ]:
# Load digits dataset
digits = load_digits()
X = digits.data / 16.0  # Normalize to [0, 1]
y = digits.target

print(f"Dataset: {X.shape}")
print(f"Samples: {len(X)}")
print(f"Features: {X.shape[1]}")

# Convert to tensor
X_tensor = torch.FloatTensor(X)

# Visualize original digits
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'Digit: {y[i]}')
    ax.axis('off')
plt.suptitle('Original Digits', fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Data loaded!')

## Step 3: Define Loss Function

**Total Loss** = MSE(x, x_recon) + KL(q(z|x) || p(z))

where KL = -0.5 * Σ(1 + log(σ²) - μ² - σ²)

In [ ]:
def vae_loss(recon_x, x, mu, logvar):
    """VAE loss = reconstruction + KL divergence"""
    # Reconstruction loss (MSE)
    recon_loss = F.mse_loss(recon_x, x, reduction='sum')
    
    # KL divergence
    # KL(N(μ,σ²) || N(0,1)) = -0.5 * Σ(1 + log(σ²) - μ² - σ²)
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return recon_loss + kld, recon_loss, kld

# Test loss
recon, mu, logvar = vae(X_tensor[:10])
total, recon_l, kld_l = vae_loss(recon, X_tensor[:10], mu, logvar)
print(f"Loss components (before training):")
print(f"  Reconstruction: {recon_l.item():.2f}")
print(f"  KL Divergence: {kld_l.item():.2f}")
print(f"  Total: {total.item():.2f}")
print('✅ Loss function defined!')

## Step 4: Train VAE

In [ ]:
# Training setup
optimizer = optim.Adam(vae.parameters(), lr=0.001)
n_epochs = 100

losses = {'total': [], 'recon': [], 'kld': []}

print("Training VAE...")
for epoch in range(n_epochs):
    # Forward pass
    recon, mu, logvar = vae(X_tensor)
    total_loss, recon_loss, kld_loss = vae_loss(recon, X_tensor, mu, logvar)
    
    # Backward pass
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    
    # Log
    losses['total'].append(total_loss.item())
    losses['recon'].append(recon_loss.item())
    losses['kld'].append(kld_loss.item())
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}')
        print(f'  Total Loss: {total_loss.item():.0f}')
        print(f'  Recon: {recon_loss.item():.0f}, KLD: {kld_loss.item():.0f}')

print('✅ VAE trained!')

## Step 5: Visualize Training Progress

In [ ]:
# Plot losses
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(losses['total'], label='Total Loss', linewidth=2)
ax.plot(losses['recon'], label='Reconstruction', linewidth=2, alpha=0.7)
ax.plot(losses['kld'], label='KL Divergence', linewidth=2, alpha=0.7)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('VAE Training Loss', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# Reconstruction quality
vae.eval()
with torch.no_grad():
    recon, _, _ = vae(X_tensor[:10])
    recon = recon.numpy()

ax = axes[1]
original_flat = X[:10].flatten()
recon_flat = recon.flatten()
ax.scatter(original_flat, recon_flat, alpha=0.5, s=10)
ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect reconstruction')
ax.set_xlabel('Original pixel value', fontsize=12)
ax.set_ylabel('Reconstructed pixel value', fontsize=12)
ax.set_title('Reconstruction Quality', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('✅ Training visualization complete!')

## Step 6: Visualize Reconstructions

In [ ]:
# Compare original vs reconstructed
fig, axes = plt.subplots(4, 10, figsize=(15, 6))

with torch.no_grad():
    recon, _, _ = vae(X_tensor[:10])
    recon = recon.numpy()

for i in range(10):
    # Original
    axes[0, i].imshow(X[i].reshape(8, 8), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Original', fontsize=11)
    
    # Reconstructed
    axes[1, i].imshow(recon[i].reshape(8, 8), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Reconstructed', fontsize=11)
    
    # Difference
    diff = np.abs(X[i] - recon[i]).reshape(8, 8)
    axes[2, i].imshow(diff, cmap='hot')
    axes[2, i].axis('off')
    if i == 0:
        axes[2, i].set_ylabel('Difference', fontsize=11)
    
    # Labels
    axes[3, i].text(0.5, 0.5, str(y[i]), fontsize=16, ha='center', va='center')
    axes[3, i].axis('off')
    if i == 0:
        axes[3, i].set_ylabel('Label', fontsize=11)

plt.suptitle('VAE Reconstructions', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print('✅ Reconstructions visualized!')

## Step 7: Generate New Samples

Sample from latent space N(0, I) to generate new digits!

In [ ]:
# Generate new samples
n_samples = 10
z_samples = torch.randn(n_samples, 8)  # Sample from N(0, I)

with torch.no_grad():
    generated = vae.decode(z_samples).numpy()

# Visualize
fig, axes = plt.subplots(1, n_samples, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(generated[i].reshape(8, 8), cmap='gray')
    ax.axis('off')
plt.suptitle('Generated Digits from Random Latent Codes', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()
print('✅ New digits generated!')

## Summary

**VAE Achievements**:
- ✅ Learned smooth latent space representation
- ✅ Can reconstruct original digits
- ✅ Can generate new plausible digits
- ✅ Latent space is regularized (follows N(0, I))

**Production Tips**:
1. **β-VAE**: Add weight β to KLD for disentangled representations
2. **Annealing**: Start with low KLD weight, increase gradually
3. **Architecture**: Use convolutional layers for images
4. **Applications**: Anomaly detection, data augmentation, compression

**VAE vs AE**:
- VAE: Probabilistic, smooth latent space, can generate
- AE: Deterministic, may have gaps in latent space